<a href="https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Eng-Waheedullah-wazir/flyrank-waheed-ml-intern/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*


I use Logistic Regression as the first predictive model for this lane.

The target is a binary decline-related outcome, so Logistic Regression is a suitable classification method. It is simple, fast, and interpretable, which makes it useful for establishing whether the selected features contain predictive signal before using a more complex model.

I use the five features defined in the earlier data contract:

- impressions_90d
- clicks_90d
- ctr
- avg_position
- scroll_rate

The model does not use trend_direction, trend_pct, or is_declining_label as input features because these fields are derived from or directly represent the target.

The purpose is not to maximize complexity. The model must demonstrate useful improvement over the Week-4 hand-written baseline on the same held-out data.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
)

df = pd.read_csv('https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv')
print("Dataset shape:", df.shape)
print("Number of clients:", df["client_id"].nunique())


Dataset shape: (30000, 44)
Number of clients: 32


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*


I use an 80/20 grouped train/test split based on client_id.

The dataset contains multiple content items for each pseudonymized client. A normal random row split could put content from the same client in both training and test data. That can make evaluation look better than it really is because the model has already seen the client's data distribution.

A grouped split keeps every client entirely in either the training or test set.

The test clients are therefore unseen during model training.

The Week-4 baseline is also evaluated on these exact same test rows so that the model-versus-baseline comparison is fair.

The test set is not used to fit the model, calculate imputation values, or fit the scaler.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Create the observed decline proxy target.
# 1 = observed downward trend
# 0 = otherwise

df["target"] = (
    df["trend_direction"] == "down"
).astype(int)

FEATURES = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "scroll_rate",
]

# Explicit leakage check
LEAKAGE_FIELDS = {
    "trend_direction",
    "trend_pct",
    "is_declining_label",
}

assert LEAKAGE_FIELDS.isdisjoint(FEATURES)

# Grouped 80/20 split
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_idx, test_idx = next(
    gss.split(
        df,
        y=df["target"],
        groups=df["client_id"],
    )
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Train rows:", len(train_df))
print("Test rows:", len(test_df))

print("Train clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

client_overlap = (
    set(train_df["client_id"])
    & set(test_df["client_id"])
)

print("Client overlap:", len(client_overlap))

assert len(client_overlap) == 0


Train rows: 23837
Test rows: 6163
Train clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*


I train Logistic Regression using only the training clients.

Missing feature values are handled inside the training pipeline using median imputation, followed by standardization. These preprocessing steps are fitted only on the training data.

The model is then evaluated on the held-out test clients.

The Week-4 baseline is recreated using the same baseline rule and evaluated on exactly the same test rows.

The primary comparison metric is F1 because the task is a binary classification problem and F1 balances precision and recall.

I also report precision, recall, ROC-AUC, and PR-AUC for additional context.

In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

X_train = train_df[FEATURES].copy()
X_test = test_df[FEATURES].copy()

y_train = train_df["target"]
y_test = test_df["target"]

print("Training features:", X_train.shape)
print("Test features:", X_test.shape)

print("\nTraining target:")
print(y_train.value_counts())

print("\nTest target:")
print(y_test.value_counts())

model = Pipeline([
    (
        "imputer",
        SimpleImputer(strategy="median")
    ),
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000,
            random_state=42
        )
    ),
])

model.fit(X_train, y_train)

print("Logistic Regression trained successfully.")

model_pred = model.predict(X_test)

model_prob = model.predict_proba(X_test)[:, 1]

model_metrics = {
    "accuracy": accuracy_score(
        y_test,
        model_pred
    ),
    "precision": precision_score(
        y_test,
        model_pred,
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        model_pred,
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        model_pred,
        zero_division=0
    ),
    "roc_auc": roc_auc_score(
        y_test,
        model_prob
    ),
    "pr_auc": average_precision_score(
        y_test,
        model_prob
    ),
}

print("Logistic Regression metrics:")

for name, value in model_metrics.items():
    print(f"{name}: {value:.4f}")



Training features: (23837, 5)
Test features: (6163, 5)

Training target:
target
1    13113
0    10724
Name: count, dtype: int64

Test target:
target
1    3149
0    3014
Name: count, dtype: int64
Logistic Regression trained successfully.
Logistic Regression metrics:
accuracy: 0.5186
precision: 0.5151
recall: 0.9854
f1: 0.6766
roc_auc: 0.5572
pr_auc: 0.5441


Recreate the Week-4 baseline

Use the exact rule you actually committed in ML-07.

Do not modify the baseline just to make the Week-5 model look better.

If your ML-07 rule used staleness plus impressions, use:

In [9]:
# Recreate the Week-4 baseline on the test rows.

baseline_test = test_df.copy()

# Use the training data to establish the visibility threshold.
# This prevents using test data to determine a modeling threshold.
visibility_threshold = train_df["impressions_90d"].quantile(0.75)

baseline_test["baseline_score"] = 0

# Staleness component
baseline_test.loc[
    baseline_test["days_since_last_update"] > 365,
    "baseline_score"
] += 3

baseline_test.loc[
    baseline_test["days_since_last_update"].between(
        181,
        365,
        inclusive="both"
    ),
    "baseline_score"
] += 2

baseline_test.loc[
    baseline_test["days_since_last_update"].between(
        91,
        180,
        inclusive="both"
    ),
    "baseline_score"
] += 1

# High visibility component
baseline_test.loc[
    baseline_test["impressions_90d"] >= visibility_threshold,
    "baseline_score"
] += 2

# Same action threshold as Week-4
baseline_test["baseline_pred"] = (
    baseline_test["baseline_score"] >= 3
).astype(int)

print(
    baseline_test["baseline_pred"].value_counts()
)

baseline_pred
0    5999
1     164
Name: count, dtype: int64


Base line matric

In [10]:
baseline_metrics = {
    "accuracy": accuracy_score(
        y_test,
        baseline_test["baseline_pred"]
    ),
    "precision": precision_score(
        y_test,
        baseline_test["baseline_pred"],
        zero_division=0
    ),
    "recall": recall_score(
        y_test,
        baseline_test["baseline_pred"],
        zero_division=0
    ),
    "f1": f1_score(
        y_test,
        baseline_test["baseline_pred"],
        zero_division=0
    ),
}

print("Week-4 baseline metrics:")

for name, value in baseline_metrics.items():
    print(f"{name}: {value:.4f}")

Week-4 baseline metrics:
accuracy: 0.4803
precision: 0.3354
recall: 0.0175
f1: 0.0332


Model-vs-baseline table

In [11]:
comparison = pd.DataFrame([
    {
        "method": "Week-4 baseline",
        "accuracy": baseline_metrics["accuracy"],
        "precision": baseline_metrics["precision"],
        "recall": baseline_metrics["recall"],
        "f1": baseline_metrics["f1"],
    },
    {
        "method": "Logistic Regression",
        "accuracy": model_metrics["accuracy"],
        "precision": model_metrics["precision"],
        "recall": model_metrics["recall"],
        "f1": model_metrics["f1"],
    },
])

print(comparison.to_string(index=False))

             method  accuracy  precision   recall       f1
    Week-4 baseline  0.480286   0.335366 0.017466 0.033203
Logistic Regression  0.518579   0.515106 0.985392 0.676551


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

cm = confusion_matrix(
    y_test,
    model_pred
)

print("Confusion matrix:")
print(cm)

tn, fp, fn, tp = cm.ravel()

print("\nTrue negatives :", tn)
print("False positives:", fp)
print("False negatives:", fn)
print("True positives :", tp)


Confusion matrix:
[[  93 2921]
 [  46 3103]]

True negatives : 93
False positives: 2921
False negatives: 46
True positives : 3103


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.